# 06b. Case study: the summer 2012 central U.S. heat wave: experiment QC

Ensemble-mean absolute error of regional-mean TREFHT on lead day 1 and week 1, every configuration. A defective initial state (`ic_defect`) shows as a cold error of about 5-9 K at every start, far beyond the event-initialized DA configurations (within about 1 K); those configurations are kept out of every later score, and their errors here say nothing about the physics of their initialization. CLIM_BASE also has large day-1 errors at some starts, as expected: its atmospheric reference state is not initialized to the 2012 weather (late May 2012 was itself unusually warm). The QC series figure shows all configurations against observations (absolute error reference).

**The 06a-06h series** (summer 2012 central U.S. heat wave; each notebook reads the data it needs, computes, saves, then plots): `06a` observed event and simulation coverage; `06b` experiment QC; `06c` start-resolved regional trajectories; `06d` fixed-event-window metrics and maps; `06e` S2S lead-window scores; `06f` event probability; `06g` initialization contrasts; `06h` sensitivity of the event definition. Settings: `HeatwaveSettings` in `configs/hind_lead_eval_cases.py`; products: `util/heatwave_workflow.py`; methods: `util/heatwave_case.py`. Design, event definition and reference states: `06a`.

Every product is saved with the settings it was computed from and read back on the next run when those settings are unchanged; `FORCE_COMPUTE = True` recomputes this notebook's results, `FORCE_INPUTS = True` also re-reads the model output and observations. Model output is read only when a product has to be (re)computed.

## Outputs

`data/hind_lead_eval/heatwave_2012/init_qc.csv`; figure `heatwave_series_qc_all_none.png`.

## 1. Libraries

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

## 2. User setup

Paths and run options; plain values only, so `papermill -p` overrides work. Case definitions (experiments, regions, windows, options) are the settings defaults; change them for this notebook with `CASE_OVERRIDES`.

In [2]:
INPUT_DATA_ROOT = Path("/global/cfs/cdirs/m4849/zhan391/eam_hindcast")
WORK_DIR = Path("/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026")
OBSERVATION_ROOT = Path("/global/cfs/cdirs/e3sm/zhan391/data")  # reference-data archive
LAND_SUPPORT_DIR = OBSERVATION_ROOT / "lnd_sea_mask"           # land fraction
AMIP_ROOT = OBSERVATION_ROOT / "E3SMv3"                        # free-running E3SMv3 DECK AMIP runs
WORKFLOW_NAME = "hind_lead_eval"

# Initialization configurations: every experiment (label, initial states, members, case name) is in
# configs/fcst_init_experiment_config.py (EXPERIMENTS). CONFIG_GROUP picks the set analysed, by question
# (CONFIG_GROUPS): "tracer_init" (a, incomplete vs completed ERA5 state), "lnd_vs_atm" (b, {CLIM, DART}
# atm x lnd), "dart_vs_capt" (c, {ERA5, DART} atm x {DATM, DART} lnd + CLIM), "coupling_consistency"
# (d, consistent vs mixed atm/lnd states), "all"; or a list of configuration keys.
# CASE_NAMES_OVERRIDE maps a selected key to another raw case name (INPUT_DATA_ROOT/<name>_<YYYYMMDD>00/<ENxx>).
CONFIG_GROUP = "dart_vs_capt"
CASE_NAMES_OVERRIDE = {}
# the experiments analysed are those of CONFIG_GROUP, in its order (tables and legends)

CASE_OVERRIDES = {}     # changes to the HeatwaveSettings defaults (configs/hind_lead_eval_cases.py), e.g. {"event_window": ("2012-06-27", "2012-07-07")}
FORCE_COMPUTE = False    # recompute this notebook's results even when they are saved with the same settings
FORCE_INPUTS = False     # also re-read the model output and observations (slow)


## 3. Paths and modules

In [3]:
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.fcst_init_experiment_config import STYLE_MAP, set_case_names
from configs.hind_lead_eval_cases import HeatwaveSettings, RunPaths
from util.hind_case_io import figure_saver, write_dataset, write_table
from util.heatwave_workflow import HeatwaveCase

paths = RunPaths(*(Path(p) for p in (INPUT_DATA_ROOT, WORK_DIR, DIAGNOSTIC_OUTPUT_ROOT, OBSERVATION_ROOT,
                                     LAND_SUPPORT_DIR, AMIP_ROOT)), workflow_name=WORKFLOW_NAME)
from configs.fcst_init_experiment_config import select_configs
CONFIGS = select_configs(CONFIG_GROUP, CASE_NAMES_OVERRIDE)   # {configuration key: raw case name}
set_case_names(CONFIGS)   # the readers find each configuration's cases by these names
for c, name in CONFIGS.items():
    print(f"[CASE] {c}: {name}_<YYYYMMDD>00")
case = HeatwaveCase(HeatwaveSettings(paths=paths, **{"configs": tuple(CONFIGS), **CASE_OVERRIDES}), force_compute=FORCE_COMPUTE,
                   force_inputs=FORCE_INPUTS)
cfg = case.cfg


[CASE] CLIM_BASE: F20TR_ne30pg2_r05_IcoswISC30E3r5_CLIM_BASE_<YYYYMMDD>00
[CASE] ERA5CLIMFILL: F20TR_ne30pg2_r05_IcoswISC30E3r5_ERA5CLIMFILL_<YYYYMMDD>00
[CASE] DA_ATM: F20TR_ne30pg2_r05_IcoswISC30E3r5_DA_ATM_<YYYYMMDD>00
[CASE] DA_LND_CLIMFILL: F20TR_ne30pg2_r05_IcoswISC30E3r5_DA_LND_CLIMFILL_<YYYYMMDD>00
[CASE] DA_COUPLED: F20TR_ne30pg2_r05_IcoswISC30E3r5_DA_COUPLED_<YYYYMMDD>00
[DATA] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/heatwave_2012
[FIGURE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/figure/hind_lead_eval/heatwave_2012
[SCORED] ['CLIM_BASE', 'ERA5CLIMFILL', 'DA_ATM', 'DA_LND_CLIMFILL', 'DA_COUPLED']
[QC ONLY] []


## 4. Read input

ERA5 daily-mean TREFHT and GPCP precipitation on the map box, both climatologies and the calendar-day thresholds (`observed_fields.nc`), and from them the observed events and the event window. Then the daily TREFHT (mean of the four 6-hour means) and PRECT of every member on the map box (`box_<CONFIG>_<SEASON>_<VAR>.nc`), put on calendar days, and their regional means: absolute temperature, anomalies against both references (`biases`), area fraction above the gridpoint threshold, and precipitation (daily and accumulated from the start).

In [4]:
obs = case.obs
print(f"[REGION] {cfg.region['name']}: {int(obs.mask.sum())} land points\n[CLIM] observed: {obs.clim_sources[0]}\n"
      f"[CLIM] model: {obs.clim_sources[1]}")
series = case.series

[READ] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/heatwave_2012/observed_fields.nc
[EVENT] window 2012-06-27 to 2012-07-07, observed peak 2012-07-04
[REGION] CENTRAL_US: 200 land points
[CLIM] observed: ERA5 1x1 daily: monthly means of the daily series
[CLIM] model: E3SMv3 AMIP monthly (v3.LR.amip_0101, v3.LR.amip_0151; cmip_ts (time bounds))
[MODEL] CLIM_BASE TREFHT: {'init': 4, 'member': 20, 'lead': 61, 'lat': 27, 'lon': 61}
[MODEL] CLIM_BASE PRECT: {'init': 4, 'member': 20, 'lead': 61, 'lat': 27, 'lon': 61}
[RUNS] [('CLIM_BASE', 'JJA'), ('DA_ATM', 'JJA'), ('DA_COUPLED', 'JJA'), ('DA_LND_CLIMFILL', 'JJA'), ('ERA5CLIMFILL', 'JJA')]
[STARTS] ['2012-05-16', '2012-05-21', '2012-05-26', '2012-05-31']
[MEMBERS] {'CLIM_BASE': 20, 'ERA5CLIMFILL': 20, 'DA_ATM': 20, 'DA_LND_CLIMFILL': 20, 'DA_COUPLED': 20}
[MODEL] ERA5CLIMFILL TREFHT: {'init': 4, 'member': 20, 'lead': 61, 'lat': 27, 'lon': 61}
[MODEL] ERA5CLIMFILL PRECT: {'init': 4, 'member': 20, 'lead': 61

## 5. Process and save

Ensemble-mean error of regional-mean TREFHT on lead day 1 and in week 1 (`init_qc.csv`).

In [5]:
qc = case.init_qc()
display(qc.pivot(index=["config", "status"], columns="init", values=["lead1_error", "week1_error"])
        .reindex(list(cfg.configs), level="config").round(2))

[STALE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/heatwave_2012/init_qc.csv: built with other settings, recomputing
[WRITE] /global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026/data/hind_lead_eval/heatwave_2012/init_qc.csv


lead1_error                                   \
init                    2012-05-16 2012-05-21 2012-05-26 2012-05-31   
config          status                                                
CLIM_BASE       scored       -1.17      -2.66      -7.79       2.14   
ERA5CLIMFILL    scored       -0.25      -0.33      -0.97       0.27   
DA_ATM          scored       -0.32       0.06      -0.68       0.15   
DA_LND_CLIMFILL scored       -0.31      -0.53      -1.23      -0.13   
DA_COUPLED      scored       -0.38      -0.14      -0.93      -0.24   

                       week1_error                                   
init                    2012-05-16 2012-05-21 2012-05-26 2012-05-31  
config          status                                               
CLIM_BASE       scored       -4.94      -4.51      -3.11      -0.72  
ERA5CLIMFILL    scored        0.01       0.58      -0.58      -0.22  
DA_ATM          scored       -0.26       0.18       0.05       1.40  
DA_LND_CLIMFILL scored       -0.03       0.52      -0.70      -0.51  
DA_COUPLED      scored       -0.27       0.19      -0.09       1.04

## 6. Plot and show

In [6]:
# ============================ Figure settings: edit here and rerun this cell ============================
SHOW = True          # False: save the figures without showing them
FIGURE_DPI = 200
FONT_SIZE = 18       # base font size (pt); every label of every figure scales with it
FONT_SCALES = dict(label=1.0, title=1.0, tick=0.9, legend=1.0, suptitle=1.15, cell=0.75)  # x FONT_SIZE

from util.fcst_init_plotting import set_font_size
set_font_size(FONT_SIZE, **FONT_SCALES)
save = figure_saver(case.figure_dir, dpi=FIGURE_DPI, show=SHOW)
# =========================================================================================================

from util.heatwave_case import plot_series
from util.heatwave_workflow import REFERENCE_LABELS, SERIES_YLABELS

if cfg.ic_defect:
    configs = list(cfg.configs)
    panels = {"TREFHT": {i: {c: series.anom["none", c, i] for c in configs} for i in case.inits},
              "PRECT": {i: {c: series.precip[c, i] for c in configs} for i in case.inits}}
    save(plot_series(panels, {"TREFHT": obs.a, "PRECT": series.obs_cum}, obs.window, STYLE_MAP, list(cfg.ic_defect),
                     ylabels=SERIES_YLABELS, spread=cfg.spread,
                     title=f"QC, all configurations (dashed: IC defect, not scored); {case.label}; "
                           f"{REFERENCE_LABELS['none']}"),
         "heatwave_series_qc_all_none.png")
else:
    print("[QC] no configuration flagged (ic_defect = ()): no QC figure")

[QC] no configuration flagged (ic_defect = ()): no QC figure
